# ✅ Solution 4: Deteksi Inkonsistensi Data

**Approach**: .str accessor + pd.to_numeric(errors='coerce') + regex untuk email  
**Key Insight**: Deteksi masalah dulu secara sistematis sebelum cleaning — jangan langsung 'fix' tanpa tahu ruang lingkup masalahnya.

In [ ]:
import numpy as np
import pandas as pd

data_kotor = {
    'id':            ['S001','S002','S003','S004','S005','S006','S007','S008','S009','S010'],
    'nama':          ['Andi','BUDI SANTOSO','citra','Dian Pratama','  Eko  ','Fani','GITA SARI','hendra','Indah','Joko'],
    'umur':          ['28','45','tiga puluh','52','19','35','41','28',None,'55'],
    'jenis_kelamin': ['L','Laki-laki','P','male','female','L','Perempuan','laki','p','M'],
    'rating':        [5,'4',3,'lima',4,'5',3,4,'2',5],
    'email':         ['andi@gmail.com','budi@yahoo','citra@hotmail.com','dian@gmail.com',
                      'eko.gmail.com','fani@gmail.com','gita@company.co.id',
                      'hendra@gmail.com','indah@outlook.com','joko@gov.go.id'],
    'pendapatan':    ['Rp 5.000.000','8000000','Rp8,500,000','12.000.000','3500000',
                      'Rp 9.500.000','7000000','Rp 4.250.000','6500000','15000000'],
    'kepuasan':      ['sangat puas','Puas','BIASA','puas','sangat puas','tidak puas','Biasa','PUAS','sangat tidak puas','Puas'],
}
df = pd.DataFrame(data_kotor)
print(f'Dataset: {df.shape}'); display(df)

In [ ]:
# ============================================================
# SOLUSI TODO 1: Identifikasi semua masalah
# ============================================================

print('=== Missing Values ===')
print(df.isnull().sum())
print()

print('=== jenis_kelamin: value_counts ===')
print(df['jenis_kelamin'].value_counts())
print(f'→ Harusnya 2 kategori (L/P), tapi ada {df["jenis_kelamin"].nunique()} nilai unik!')
print()

print('=== kepuasan: value_counts ===')
print(df['kepuasan'].value_counts())
print(f'→ Case sensitivity: "PUAS", "Puas", "puas" seharusnya sama!')
print(f'→ Nilai unik: {df["kepuasan"].nunique()} (harusnya ~5)')

In [ ]:
# ============================================================
# SOLUSI TODO 2: Deteksi email tidak valid
# ============================================================

# Email valid: harus punya '@' DAN bagian setelah '@' harus ada '.'
# Pattern: ada '@' + setelah '@' ada minimal satu '.'
def is_valid_email(email):
    if '@' not in email:
        return False
    local, domain = email.split('@', 1)
    return '.' in domain

mask_tidak_valid = ~df['email'].apply(is_valid_email)

print('Email tidak valid:')
print(df[mask_tidak_valid][['id','email']])
print()
print('Analisis:')
print('  budi@yahoo    → domain "yahoo" tidak punya "." (bukan .com/.net/dsb)')
print('  eko.gmail.com → tidak punya "@", hanya ada "."')

In [ ]:
# ============================================================
# SOLUSI TODO 3: Deteksi umur yang tidak bisa dikonversi
# ============================================================

umur_converted = pd.to_numeric(df['umur'], errors='coerce')
mask_umur_error = umur_converted.isnull()

print('Baris dengan umur bermasalah:')
print(df[mask_umur_error][['id','nama','umur']])
print()
print('Penjelasan:')
print('  S003 → "tiga puluh" : teks bukan angka')
print('  S009 → None         : missing value')

# Juga cek rating
rating_converted = pd.to_numeric(df['rating'], errors='coerce')
mask_rating_error = rating_converted.isnull()
print()
print('Baris dengan rating bermasalah:')
print(df[mask_rating_error][['id','nama','rating']])

In [ ]:
# ============================================================
# SOLUSI TODO 4: Data Quality Report
# ============================================================

n_total = len(df)

laporan = pd.DataFrame({
    'kolom': [
        'umur', 'jenis_kelamin', 'rating', 'email',
        'kepuasan', 'nama', 'pendapatan'
    ],
    'masalah': [
        'Nilai teks ("tiga puluh") dan missing value (None)',
        'Inkonsistensi format: L/P, Laki-laki/Perempuan, male/female, laki, M, p',
        'Tipe campuran: int dan string ("lima", "4")',
        'Format tidak valid: tidak ada @, atau domain tanpa titik',
        'Inkonsistensi kapitalisasi: PUAS, Puas, puas dihitung beda',
        'Inkonsistensi kapitalisasi dan whitespace: "  Eko  ", "BUDI SANTOSO", "citra"',
        'Format angka berbeda: "Rp 5.000.000", "8000000", "Rp8,500,000"',
    ],
    'jumlah_baris_bermasalah': [
        2,   # S003 (teks) + S009 (None)
        8,   # hampir semua kecuali yang format standar
        1,   # S004 'lima'
        2,   # budi@yahoo + eko.gmail.com
        10,  # semua baris karena casing tidak konsisten
        10,  # semua baris
        7,   # 7 dari 10 pakai format non-standar
    ],
    'persentase': [
        round(2/n_total*100, 1),
        round(8/n_total*100, 1),
        round(1/n_total*100, 1),
        round(2/n_total*100, 1),
        round(10/n_total*100, 1),
        round(10/n_total*100, 1),
        round(7/n_total*100, 1),
    ]
})

print('=== DATA QUALITY REPORT ===')
display(laporan)
print(f'\nTotal kolom bermasalah: {len(laporan)} dari {len(df.columns)}')

## 📌 Takeaways

- `pd.to_numeric(series, errors='coerce')` → konversi ke angka, nilai gagal jadi NaN
- `.str.contains(pattern)` → cek pattern dalam string (mendukung regex)
- `.nunique()` → jumlah nilai unik dalam kolom
- `.value_counts()` → cara cepat lihat inkonsistensi nilai kategorik
- **Prioritas masalah**: tipe data salah > missing value > inkonsistensi format > casing
- Selalu buat Data Quality Report sebelum mulai cleaning